In [0]:
--SELECT version();
--USE DATABASE qdp_database;

CREATE SCHEMA IF NOT EXISTS qdp_medication_request;
SET search_path TO qdp_medication_request;


-- metadata table
DROP TABLE IF EXISTS metadata;


CREATE TABLE metadata (
	supplier VARCHAR,
	data_product VARCHAR,
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );



INSERT INTO metadata
(
  supplier,
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
)
VALUES (
  'quantexa.com',
  'com.quantexa.qdp_medication_request',
  '0.1',
  'medication_request.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);




-- -----------------------------------------------------------
-- create hub_medication_request table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS hub_medication_request CASCADE;

CREATE TABLE hub_medication_request (
    medication_request_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_request_entity_id VARCHAR,
  	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE hub_medication_request ADD CONSTRAINT pk_medrequest_hubmedrequest_medrequestid PRIMARY KEY (medication_request_id);

COMMENT ON COLUMN hub_medication_request.medication_request_id IS 'Medication Request Identity';




-- -----------------------------------------------------------
-- create sat_medication_request table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request;

CREATE TABLE sat_medication_request (
    sat_medication_request_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
    medication_request_id BIGINT NOT NULL,
    load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    record_source_id BIGINT NOT NULL DEFAULT 1,
	data_source_code VARCHAR,
	data_source_concept_id BIGINT,
	data_source_raw_code VARCHAR,
	data_source_raw_concept_id BIGINT,
	medication_prescription_id BIGINT,
	prescription_ref VARCHAR,
	status_code VARCHAR,
	status_concept_id BIGINT,
	status_raw_code VARCHAR,
	status_raw_concept_id BIGINT,
	status_reason_code VARCHAR,
	status_reason_concept_id BIGINT,
	status_reason_raw_code VARCHAR,
	status_reason_raw_concept_id BIGINT,
	status_changed TIMESTAMP,
	intent_code VARCHAR,
	intent_concept_id BIGINT,
	intent_raw_code VARCHAR,
	intent_raw_concept_id BIGINT,
	priority_code VARCHAR,
	priority_concept_id BIGINT,
	priority_raw_code VARCHAR,
	priority_raw_concept_id BIGINT,
	do_not_perform BOOLEAN,
	medication_code VARCHAR,
	medication_concept_id BIGINT,
	medication_raw_code VARCHAR,
	medication_raw_concept_id BIGINT,
	subject_party_id BIGINT,
	authored_on TIMESTAMP,
	requester_party_id BIGINT,
	reported BOOLEAN,
	performer_type_code VARCHAR,
	performer_type_concept_id BIGINT,
	performer_type_raw_code VARCHAR,
	performer_type_raw_concept_id BIGINT,
	recorder_role_id BIGINT,
	course_of_therapy_type_code VARCHAR,
	course_of_therapy_type_concept_id BIGINT,
	course_of_therapy_type_raw_code VARCHAR,
	course_of_therapy_type_raw_concept_id BIGINT,
	rendered_dosage_instruction TEXT,
	effective_dose_period_start TIMESTAMP,
	effective_dose_period_end TIMESTAMP,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request ADD CONSTRAINT pk_medrequest_satmedrequest_satmedrequestid PRIMARY KEY (sat_medication_request_id);
ALTER TABLE sat_medication_request ADD CONSTRAINT fk_medrequest_satmedrequest_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request.sat_medication_request_id IS 'Medication Request Identity Sat';
COMMENT ON COLUMN sat_medication_request.medication_prescription_id IS 'Medication Prescription Identity. Cross-schema reference to qdp_medication_prescriptions';
COMMENT ON COLUMN sat_medication_request.prescription_ref IS 'The GUID Reference Identifier for the Issued Prescription';
COMMENT ON COLUMN sat_medication_request.status_code IS 'Current status of the Prescription Medication Line. E.g. active, on-hold, ended, stopped, completed, cancelled, entered-in-error, draft, unknown. See https://hl7.org/fhir/R5/valueset-medicationrequest-status.html';
COMMENT ON COLUMN sat_medication_request.status_reason_code IS 'Reason for current status. E.g. altchoice, clarif, drughigh, hospadm, labint, non-avail, preg, salg, sddi, sdupther, sintol, surg, washout. See https://build.fhir.org/valueset-medicationrequest-status-reason.html';
COMMENT ON COLUMN sat_medication_request.status_changed IS 'When the status was changed';
COMMENT ON COLUMN sat_medication_request.intent_code IS 'Whether the request is a proposal, plan, or an original order. E.g. proposal, plan, order, original-order, reflex-order, filler-order, instance-order, option. See https://hl7.org/fhir/R5/valueset-medicationrequest-intent.html';
COMMENT ON COLUMN sat_medication_request.priority_code IS 'Indicates how quickly the Medication Request should be addressed. E.g. routine, urgent, asap, stat. See https://hl7.org/fhir/R5/valueset-request-priority.html';
COMMENT ON COLUMN sat_medication_request.do_not_perform IS 'True if patient is to stop taking or not to start taking the medication';
COMMENT ON COLUMN sat_medication_request.medication_code IS 'What medication has been Requested (SNOMED CT coded). See https://build.fhir.org/valueset-medication-codes.html';
COMMENT ON COLUMN sat_medication_request.subject_party_id IS 'Individual or group for whom the medication has been requested';
COMMENT ON COLUMN sat_medication_request.authored_on IS 'When request was initially authored. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.authoredOn';
COMMENT ON COLUMN sat_medication_request.requester_party_id IS 'Who/What requested the Request. E.g. Practitioner, PractitionerRole, Organization, Patient, RelatedPerson, Device';
COMMENT ON COLUMN sat_medication_request.reported IS 'Reported rather than primary record. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.reported';
COMMENT ON COLUMN sat_medication_request.performer_type_code IS 'Desired kind of performer of the medication administration. E.g. registerednurse, oncologynurse, paincontrolnurse, physician, pharmacist. See https://build.fhir.org/valueset-medication-intended-performer-role.html';
COMMENT ON COLUMN sat_medication_request.recorder_role_id IS 'Person who entered the request. E.g. Practitioner, PractitionerRole';
COMMENT ON COLUMN sat_medication_request.course_of_therapy_type_code IS 'Overall pattern of medication administration. E.g. continuous, acute, seasonal. See https://build.fhir.org/valueset-medicationrequest-course-of-therapy.html';
COMMENT ON COLUMN sat_medication_request.rendered_dosage_instruction IS 'Specific instructions for how the medication is to be used by the patient';
COMMENT ON COLUMN sat_medication_request.effective_dose_period_start IS 'Start of Period over which the medication is to be taken';
COMMENT ON COLUMN sat_medication_request.effective_dose_period_end IS 'End of Period over which the medication is to be taken';



-- -----------------------------------------------------------
-- create sat_medication_request_category table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_category;

CREATE TABLE sat_medication_request_category (
	medication_request_category_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	category_code VARCHAR,
	category_concept_id BIGINT,
	category_raw_code VARCHAR,
	category_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_category ADD CONSTRAINT pk_medrequest_satmedrequestcategory_medrequestcategoryid PRIMARY KEY (medication_request_category_id);
ALTER TABLE sat_medication_request_category ADD CONSTRAINT fk_medrequest_satmedrequestcategory_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_category.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_category.medication_request_category_id IS 'Medication Request Category Identity';
COMMENT ON COLUMN sat_medication_request_category.category_code IS 'Grouping or category of medication request. E.g. inpatient, outpatient, community. See https://hl7.org/fhir/R5/valueset-medicationdispense-admin-location.html';
COMMENT ON COLUMN sat_medication_request_category.rank IS 'The recommended order the Medication Request Category Codes should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_device table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_device;

CREATE TABLE sat_medication_request_device (
	medication_request_device_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	device_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_device ADD CONSTRAINT pk_medrequest_satmedrequestdevice_medrequestdeviceid PRIMARY KEY (medication_request_device_id);
ALTER TABLE sat_medication_request_device ADD CONSTRAINT fk_medrequest_satmedrequestdevice_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_device.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_device.medication_request_device_id IS 'Medication Request Device Identity';
COMMENT ON COLUMN sat_medication_request_device.device_id IS 'Unique Identifier for the Device. See https://build.fhir.org/devicedefinition.html';
COMMENT ON COLUMN sat_medication_request_device.rank IS 'The recommended order the Medication Request Devices should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_dosage_instructions table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_dosage_instructions;

CREATE TABLE sat_medication_request_dosage_instructions (
	medication_request_dosage_instructions_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	dosage_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_dosage_instructions ADD CONSTRAINT pk_medrequest_satmedrequestdosageinstr_medrequestdosageinstrid PRIMARY KEY (medication_request_dosage_instructions_id);
ALTER TABLE sat_medication_request_dosage_instructions ADD CONSTRAINT fk_medrequest_satmedrequestdosageinstr_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_dosage_instructions.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_dosage_instructions.medication_request_dosage_instructions_id IS 'Medication Request Dosage Instructions Identity';
COMMENT ON COLUMN sat_medication_request_dosage_instructions.dosage_id IS 'Unique Identifier for the Dosage Instructions. See https://build.fhir.org/dosage.html#Dosage';
COMMENT ON COLUMN sat_medication_request_dosage_instructions.rank IS 'The recommended order the Medication Request Dosage Instructions should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_information_source table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_information_source;

CREATE TABLE sat_medication_request_information_source (
	medication_request_information_source_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	information_source_party_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_information_source ADD CONSTRAINT pk_medrequest_satmedrequestinfosource_medrequestinfosourceid PRIMARY KEY (medication_request_information_source_id);
ALTER TABLE sat_medication_request_information_source ADD CONSTRAINT fk_medrequest_satmedrequestinfosource_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_information_source.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_information_source.medication_request_information_source_id IS 'Medication Request Information Source Identity';
COMMENT ON COLUMN sat_medication_request_information_source.information_source_party_id IS 'The Party who supplied this Information about the Request. E.g. Patient, Practitioner, PractitionerRole, RelatedPerson, Organization. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.informationSource';
COMMENT ON COLUMN sat_medication_request_information_source.rank IS 'The recommended order the Medication Request Information Sources should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_notes table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_notes;

CREATE TABLE sat_medication_request_notes (
	medication_request_note_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	author_reference_party_id BIGINT,
	author_string VARCHAR,
	time TIMESTAMP,
	text TEXT NOT NULL,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_notes ADD CONSTRAINT pk_medrequest_satmedrequestnotes_medrequestnoteid PRIMARY KEY (medication_request_note_id);
ALTER TABLE sat_medication_request_notes ADD CONSTRAINT fk_medrequest_satmedrequestnotes_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_notes.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_notes.medication_request_note_id IS 'Medication Request Note Identity';
COMMENT ON COLUMN sat_medication_request_notes.author_reference_party_id IS 'The clinical Party Identifier that recorded this information. E.g. Practitioner, PractitionerRole, Patient, RelatedPerson, Organization. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.note';
COMMENT ON COLUMN sat_medication_request_notes.author_string IS 'The Individual that recorded this information';
COMMENT ON COLUMN sat_medication_request_notes.time IS 'When this information was recorded';
COMMENT ON COLUMN sat_medication_request_notes.text IS 'Clinician entered additional information about the Medication Request';



-- -----------------------------------------------------------
-- create sat_medication_request_performer table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_performer;

CREATE TABLE sat_medication_request_performer (
	medication_request_performer_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	performer_party_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_performer ADD CONSTRAINT pk_medrequest_satmedrequestperformer_medrequestperformerid PRIMARY KEY (medication_request_performer_id);
ALTER TABLE sat_medication_request_performer ADD CONSTRAINT fk_medrequest_satmedrequestperformer_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_performer.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_performer.medication_request_performer_id IS 'Medication Request Performer Identity';
COMMENT ON COLUMN sat_medication_request_performer.performer_party_id IS 'Intended performer of administration. E.g. Patient, Practitioner, PractitionerRole, RelatedPerson, Organization, DeviceDefinition, CareTeam, HealthcareService. See https://build.fhir.org/medicationrequest-definitions.html#MedicationRequest.performer';
COMMENT ON COLUMN sat_medication_request_performer.rank IS 'The recommended order the Medication Request Performers should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_reason table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_reason;

CREATE TABLE sat_medication_request_reason (
	medication_request_reason_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	reason_code VARCHAR,
	reason_concept_id BIGINT,
	reason_raw_code VARCHAR,
	reason_raw_concept_id BIGINT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_reason ADD CONSTRAINT pk_medrequest_satmedrequestreason_medrequestreasonid PRIMARY KEY (medication_request_reason_id);
ALTER TABLE sat_medication_request_reason ADD CONSTRAINT fk_medrequest_satmedrequestreason_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_reason.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_reason.medication_request_reason_id IS 'Medication Request Reason Identity';
COMMENT ON COLUMN sat_medication_request_reason.reason_code IS 'Reason or indication for ordering or not ordering the medication (SNOMED CT clinical findings coded). See https://build.fhir.org/valueset-condition-code.html';
COMMENT ON COLUMN sat_medication_request_reason.rank IS 'The recommended order the Medication Request Reason Codes should be presented';



-- -----------------------------------------------------------
-- create sat_medication_request_supporting_information table
-- -----------------------------------------------------------
DROP TABLE IF EXISTS sat_medication_request_supporting_information;

CREATE TABLE sat_medication_request_supporting_information (
	medication_request_supporting_information_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (MINVALUE 0 START WITH 0 INCREMENT BY 1),
	medication_request_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	identifier_type VARCHAR,
	identifier BIGINT,
	reference VARCHAR,
	notes TEXT,
	rank INT,
	period_start TIMESTAMP,
	period_end TIMESTAMP
);

ALTER TABLE sat_medication_request_supporting_information ADD CONSTRAINT pk_medrequest_satmedrequestsuppinfo_medrequestsuppinfoid PRIMARY KEY (medication_request_supporting_information_id);
ALTER TABLE sat_medication_request_supporting_information ADD CONSTRAINT fk_medrequest_satmedrequestsuppinfo_hubmedrequest_medrequestid FOREIGN KEY (medication_request_id) REFERENCES hub_medication_request(medication_request_id);

COMMENT ON COLUMN sat_medication_request_supporting_information.medication_request_id IS 'Medication Request Identity';
COMMENT ON COLUMN sat_medication_request_supporting_information.medication_request_supporting_information_id IS 'Medication Request Supporting Information Identity';
COMMENT ON COLUMN sat_medication_request_supporting_information.identifier_type IS 'Type the reference refers to (e.g. "Patient") - must be a resource in resources. See https://hl7.org/fhir/R5/valueset-resource-types.html';
COMMENT ON COLUMN sat_medication_request_supporting_information.identifier IS 'Logical reference, when literal reference is not known';
COMMENT ON COLUMN sat_medication_request_supporting_information.reference IS 'Literal reference, Relative, internal or absolute URL';
COMMENT ON COLUMN sat_medication_request_supporting_information.notes IS 'Human Actor entered supporting information';
COMMENT ON COLUMN sat_medication_request_supporting_information.rank IS 'The recommended order the Medication Request Supporting Information Entries should be presented';




-- -----------------------------------------------------------
-- create view_medication_requests view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_requests AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_request_id,
	h.medication_request_entity_id,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
  s.sat_medication_request_id,
  s.medication_prescription_id,
  s.prescription_ref,
  s.status_code,
  s.status_concept_id,
  s.status_raw_code,
  s.status_raw_concept_id,
  s.status_reason_code,
  s.status_reason_concept_id,
  s.status_reason_raw_code,
  s.status_reason_raw_concept_id,
  s.status_changed,
  s.intent_code,
  s.intent_concept_id,
  s.intent_raw_code,
  s.intent_raw_concept_id,
  s.priority_code,
  s.priority_concept_id,
  s.priority_raw_code,
  s.priority_raw_concept_id,
  s.do_not_perform,
  s.medication_code,
  s.medication_concept_id,
  s.medication_raw_code,
  s.medication_raw_concept_id,
  s.subject_party_id,
  s.authored_on,
  s.requester_party_id,
  s.reported,
  s.performer_type_code,
  s.performer_type_concept_id,
  s.performer_type_raw_code,
  s.performer_type_raw_concept_id,
  s.recorder_role_id,
  s.course_of_therapy_type_code,
  s.course_of_therapy_type_concept_id,
  s.course_of_therapy_type_raw_code,
  s.course_of_therapy_type_raw_concept_id,
  s.rendered_dosage_instruction,
  s.effective_dose_period_start,
  s.effective_dose_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_request.hub_medication_request h
    JOIN budd.qdp_medication_request.sat_medication_request s
      ON h.medication_request_id = s.medication_request_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_requests;


-- -----------------------------------------------------------
-- create view_medication_request_categories view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_request_categories AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_request_id,
	h.medication_request_entity_id,
  s.medication_prescription_id,
  s.status_code,
  s.medication_code,
	rc.medication_request_category_id,
	rc.category_code,
	rc.category_concept_id,
	rc.category_raw_code,
	rc.category_raw_concept_id,
	rc.rank AS category_rank,
	rc.period_start AS category_period_start,
	rc.period_end AS category_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_request.hub_medication_request h
    JOIN budd.qdp_medication_request.sat_medication_request s
      ON h.medication_request_id = s.medication_request_id
    JOIN budd.qdp_medication_request.sat_medication_request_category rc
      ON h.medication_request_id = rc.medication_request_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_request_categories;


-- -----------------------------------------------------------
-- create view_medication_request_reasons view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_request_reasons AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_request_id,
	h.medication_request_entity_id,
  s.medication_prescription_id,
  s.status_code,
  s.medication_code,
	rr.medication_request_reason_id,
	rr.reason_code,
	rr.reason_concept_id,
	rr.reason_raw_code,
	rr.reason_raw_concept_id,
	rr.rank AS reason_rank,
	rr.period_start AS reason_period_start,
	rr.period_end AS reason_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_request.hub_medication_request h
    JOIN budd.qdp_medication_request.sat_medication_request s
      ON h.medication_request_id = s.medication_request_id
    JOIN budd.qdp_medication_request.sat_medication_request_reason rr
      ON h.medication_request_id = rr.medication_request_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_request_reasons;


-- -----------------------------------------------------------
-- create view_medication_request_performers view
-- -----------------------------------------------------------
CREATE OR REPLACE VIEW view_medication_request_performers AS
SELECT
  h.tennant_id,
  t.tennant_name,
  h.medication_request_id,
	h.medication_request_entity_id,
  s.medication_prescription_id,
  s.status_code,
  s.medication_code,
	rp.medication_request_performer_id,
	rp.performer_party_id,
	rp.rank AS performer_rank,
	rp.period_start AS performer_period_start,
	rp.period_end AS performer_period_end,
	h.period_start AS hub_period_start,
	h.period_end AS hub_period_end,
	s.period_start,
	s.period_end,
	s.load_datetime,
	s.record_source_id,
	s.data_source_code,
	s.data_source_concept_id,
	s.data_source_raw_code,
	s.data_source_raw_concept_id

    FROM budd.qdp_medication_request.hub_medication_request h
    JOIN budd.qdp_medication_request.sat_medication_request s
      ON h.medication_request_id = s.medication_request_id
    JOIN budd.qdp_medication_request.sat_medication_request_performer rp
      ON h.medication_request_id = rp.medication_request_id
    JOIN budd.qdp_refcodes.tennant t
      on h.tennant_id = t.tennant_id
;

SELECT * from view_medication_request_performers;


/**

**/
